# Ingest Circuits File

1. Read the file using spark dataframe reader API
2. Add Metadata Columns 
    - Source File
    - Ingestion Timestamp
3. Write to bronze delta table

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-Common/01.environment-config

In [0]:
%run ../00-Common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

### Step 1 - Read the CSV file using the dataframe reader API

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
circuits_schema = StructType([
    StructField('cicrcuitId', StringType()),
    StructField('url', StringType()),
    StructField('cicrcuitName', StringType()),
    StructField('lat', DoubleType()),
    StructField('long', DoubleType()),
    StructField('locality', StringType()),
    StructField('country', StringType())
])

In [0]:
circuits_df = (
        spark.read.format('csv')
        .option('header', 'true')
        # .option('inferSchema','true')
        .option('mode','FAILFAST')
        .schema(circuits_schema)
        .load(source_file)
    )


### Step 2 - Add Metadata Columns

1. Source File
2. Ingestion Timestamp

In [0]:
circuits_final_df = add_ingestion_metadata(circuits_df)


### Step 3 - Write to bronze delta table

In [0]:
print(v_batch_id)

In [0]:
write_to_bronze(input_df = circuits_final_df, table_name = table_name, batch_id = v_batch_id)

In [0]:
display(spark.table(table_name))